# forcing distributions

histograms of predicted τx τy QT vs bulk truth on the held out test set. RMSE hides whether the ML matches the shape of the distribution. this checks that.

In [ ]:
using JLD2
using Plots
using Statistics

gr()
default(linewidth=1.5)

const ROOT = normpath(joinpath(@__DIR__, ".."))

v2 = JLD2.load(joinpath(ROOT, "data/generated/era5_forcing_results_v2.jld2"))
Yte = v2["Yte"]
Yte_mlp = v2["Yte_mlp"]
Yte_lin = v2["Yte_lin"]
target_names = v2["target_names"]

## truth vs MLP distribution overlay

In [ ]:
function hist_pair(j, label)
    t = Yte[:, j]
    mlp = Yte_mlp[:, j]
    lo, hi = minimum(vcat(t, mlp)), maximum(vcat(t, mlp))
    bins = range(lo, stop=hi, length=60)
    plt = histogram(t, bins=bins, alpha=0.5, color=:green, label="truth",
                    xlabel=label, ylabel="count", title="$label distribution")
    histogram!(plt, mlp, bins=bins, alpha=0.5, color=:red, label="v2 MLP")
    plt
end

plot(hist_pair(1, "τx"), hist_pair(2, "τy"), hist_pair(3, "QT"),
     layout=(3,1), size=(1000,900))

## residual distribution (prediction − truth)

In [ ]:
function resid_hist(j, label)
    r_mlp = Yte_mlp[:, j] .- Yte[:, j]
    r_lin = Yte_lin[:, j] .- Yte[:, j]
    lo, hi = -3*std(r_mlp), 3*std(r_mlp)
    bins = range(lo, stop=hi, length=60)
    plt = histogram(clamp.(r_lin, lo, hi), bins=bins, alpha=0.5, color=:blue, label="linear",
                    xlabel="pred − truth", ylabel="count", title="$label residuals")
    histogram!(plt, clamp.(r_mlp, lo, hi), bins=bins, alpha=0.5, color=:red, label="v2 MLP")
    vline!(plt, [0], color=:black, linestyle=:dash, label=nothing)
    plt
end

plot(resid_hist(1, "τx"), resid_hist(2, "τy"), resid_hist(3, "QT"),
     layout=(3,1), size=(1000,900))

## stats table — mean, std, min, max

In [ ]:
using Printf
println(rpad("target",8), " | ", rpad("truth mean±std",22), rpad("MLP mean±std",22), "MLP bias")
println("-"^70)
for j in eachindex(target_names)
    t = Yte[:, j]
    m = Yte_mlp[:, j]
    bias = mean(m) - mean(t)
    @printf("%-8s | %+7.4f ± %-9.4f %+7.4f ± %-9.4f %+7.4f\n",
            target_names[j], mean(t), std(t), mean(m), std(m), bias)
end